Đoạn mã được cung cấp là một ứng dụng Streamlit được thiết kế để xây dựng hệ thống Retrieval-Augmented Generation (RAG) sử dụng mô hình DeepSeek R1 từ Ollama. Ứng dụng cho phép người dùng tải lên tài liệu PDF, chia nhỏ thành các đoạn, tạo các biểu diễn nhúng (embeddings), và truy xuất nội dung liên quan để trả lời các câu hỏi của người dùng.  Đầu tiên, các thư viện cần thiết được nhập vào, bao gồm streamlit cho giao diện web, langchain_community cho việc tải tài liệu, nhúng và lưu trữ vector, và langchain cho các mẫu prompt và chuỗi.

In [ ]:
import streamlit as st
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains.llm import LLMChain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains import RetrievalQA

Một biến model_name được định nghĩa để lưu trữ tên mô hình, có thể dễ dàng thay đổi nếu cần. Ngoài ra, một bảng màu được định nghĩa cho giao diện người dùng của ứng dụng Streamlit để cải thiện độ tương phản và khả năng đọc.

In [ ]:
model_name = "llama3.2:3b-instruct-q8_0"
primary_color = "#007BFF"
secondary_color = "#FFC107"
background_color = "#F8F9FA"
sidebar_background = "#2C2F33"
text_color = "#212529"
sidebar_text_color = "#FFFFFF"
header_text_color = "#000000"

Bố cục chính của ứng dụng được định kiểu bằng CSS tùy chỉnh được chèn qua st.markdown. Điều này bao gồm việc định kiểu cho nền chính, thanh bên, tiêu đề, văn bản, bộ tải tệp và thanh điều hướng.

In [ ]:
st.markdown("""
    <style>
    .stApp { background-color: #F8F9FA; color: #212529; }
    [data-testid="stSidebar"] { background-color: #2C2F33 !important; color: #FFFFFF !important; }
    [data-testid="stSidebar"] * { color: #FFFFFF !important; font-size: 16px !important; }
    h1, h2, h3, h4, h5, h6 { color: #000000 !important; font-weight: bold; }
    p, span, div { color: #212529 !important; }
    .stFileUploader>div>div>div>button { background-color: #FFC107; color: #000000; font-weight: bold; border-radius: 8px; }
    header { background-color: #1E1E1E !important; }
    header * { color: #FFFFFF !important; }
    </style>
""", unsafe_allow_html=True)

Tiêu đề của ứng dụng và thanh bên được định nghĩa, với các hướng dẫn cho người dùng và các cài đặt hiển thị mô hình nhúng, loại truy xuất và mô hình LLM đang được sử dụng.

In [ ]:
st.title("📄 Build a RAG System with DeepSeek R1 & Ollama")
with st.sidebar:
    st.header("Instructions")
    st.markdown("""
    1. Upload a PDF file using the uploader below.
    2. Ask questions related to the document.
    3. The system will retrieve relevant content and provide a concise answer.
    """)
    st.header("Settings")
    st.markdown(f"""
    - **Embedding Model**: HuggingFace
    - **Retriever Type**: Similarity Search
    - **LLM**: {model_name} (Ollama)
    """)

Phần chính của ứng dụng cho phép người dùng tải lên tài liệu PDF. Khi tài liệu được tải lên, tệp sẽ được lưu cục bộ và tài liệu được tải bằng PDFPlumberLoader.

In [ ]:
uploaded_file = st.file_uploader("Upload your PDF file here", type="pdf")
if uploaded_file is not None:
    st.success("PDF uploaded successfully! Processing...")
    with open("temp.pdf", "wb") as f:
        f.write(uploaded_file.getvalue())
    loader = PDFPlumberLoader("temp.pdf")
    docs = loader.load()

Tài liệu sau đó được chia thành các đoạn bằng SemanticChunker và HuggingFaceEmbeddings. Các đoạn này được sử dụng để tạo các biểu diễn nhúng, được lưu trữ trong một FAISS vector store. Một bộ truy xuất được thiết lập để thực hiện các tìm kiếm tương tự trên các biểu diễn nhúng này.

In [ ]:
text_splitter = SemanticChunker(HuggingFaceEmbeddings())
documents = text_splitter.split_documents(docs)
embedder = HuggingFaceEmbeddings()
vector = FAISS.from_documents(documents, embedder)
retriever = vector.as_retriever(search_type="similarity", search_kwargs={"k": 3})

Mô hình LLM được khởi tạo bằng lớp Ollama với model_name đã chỉ định. Một mẫu prompt được định nghĩa để hướng dẫn LLM trong việc tạo câu trả lời dựa trên ngữ cảnh được truy xuất.

In [ ]:
llm = Ollama(model=model_name)
prompt = """
1. Use the following pieces of context to answer the question at the end.
2. If you don't know the answer, just say that "I don't know" but don't make up an answer on your own.\n
3. Keep the answer crisp and limited to 3,4 sentences.
Context: {context}
Question: {question}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(prompt)

Các chuỗi tài liệu và chuỗi kết hợp được định nghĩa bằng LLMChain và StuffDocumentsChain, tương ứng. Các chuỗi này được sử dụng để kết hợp các tài liệu được truy xuất và tạo ra câu trả lời cuối cùng.

In [ ]:
llm_chain = LLMChain(llm=llm, prompt=QA_CHAIN_PROMPT, verbose=True)
document_prompt = PromptTemplate(
    input_variables=["page_content", "source"],
    template="Context:\ncontent:{page_content}\nsource:{source}",
)
combine_documents_chain = StuffDocumentsChain(
    llm_chain=llm_chain,
    document_variable_name="context",
    document_prompt=document_prompt,
    verbose=True
)
qa = RetrievalQA(
    combine_documents_chain=combine_documents_chain,
    retriever=retriever,
    verbose=True,
    return_source_documents=True
)

Cuối cùng, ứng dụng cung cấp một trường nhập liệu cho người dùng để đặt câu hỏi liên quan đến tài liệu đã tải lên. Hệ thống xử lý truy vấn, truy xuất nội dung liên quan và hiển thị câu trả lời.

In [ ]:
st.header("❓ Ask a Question")
user_input = st.text_input("Type your question related to the document:")
if user_input:
    with st.spinner("Processing your query..."):
        try:
            response = qa(user_input)["result"]
            st.success("✅ Response:")
            st.write(response)
        except Exception as e:
            st.error(f"An error occurred: {e}")
else:
    st.info("Please upload a PDF file to start.")